## Azure ML Setup
Get the workspace

In [ ]:
from azureml.core import Workspace, Dataset, Datastore

ws = Workspace.from_config()

## Sample Dataset and explore
Use Datasets to access the data and load into a Python framework.

In [ ]:
ds = ws.datasets['noaa-isd-tabular']
df = ds.sample(1000).to_pandas_dataframe()

df.describe()

## Start Spark Session
Inputs:
* compute-target - name of attached Synapse Spark Pool
* environment - name of Azure ML Environment to use 

"Outputs":
* '%%synapse' or '%%synapse pyspark' magic for running PySpark framework 
* '%%synapse sql' for running SQL language queries on Hive/Delta tables
* '%%synapse scala' for running pure Spark (scala) code
* '%%synapse r' for running R language 

Notes:
* idle timeout set at Pool level (default: 15 mins)
* docker section of Environments will be ignored - throw warning 

Dependencies:
* Azure ML Datasets for "glue" between compute targets both interactively & in ML Pipelines
* Azure ML Environments for package & spark configuration from all Azure ML surface area (UI, APIs)
* Azure ML + DevDiv + Synapse for interactive, scalable open source data science work across compute platforms 
* Azure Synapse for integration experience 

In [ ]:
%synapse start --compute-target sparky --environment 'AzureML-Synapse' # compute target name

In [ ]:
%synapse restart --environment 'MyCustomEnvironment' # restart with new spark/python Environment changes

## Passing around state/variables/files/dataframes 
There is no state saved between your "local" compute and the remote Synapse Spark Pool. 

You should use the Workspace object to access Datasets for transferring any state, including dataframes, between compute platforms. 

In [ ]:
%%synapse # default to PySpark

# get workspace
from azureml.core import Workspace

# automagic
ws = Workspace.from_config()
ws

In [ ]:
%%synapse 

ds = ws.datasets['noaa-isd-tabular']
ds

In [ ]:
%%synapse

df = ds.to_spark_dataframe() # no sampling needed 
df.show()

## Build features for ML training
Data preparation to clean and prepare features for ML training. 

In [ ]:
%%synapse

# basic EDA
df.summary().eval() # equivalent to Pandas dataframe df.describe()

## start data prep code
X = df.drop('target')
Y = df['target']

X = X.fillna('0').groupby(df['datetime']).mean().filter(df['temperature'] < 50)
Y = Y.fillna(NaN).groupby(df['datetime']).mean()
## end data prep code

# save state in cloud as Dataset
dsX = Dataset.Tabular.from_spark_df(X).register(ws, 'noaa-isd-X')
dsY = Dataset.Tabular.from_spark_df(Y).register(ws, 'noaa-isd-Y')

## Options for saving tables (dataframes)

Users have two major options:
* save "locally" to cluster
* save to cloud

For local saves, recommend using delta tables.

For cloud saves, recommend using Azure ML Datasets. 

In [ ]:
%%synapse 

# write to detla table in temp storage for use in Synapse Spark/Dask contexts
df.write.format('delta').save(f'/tmp/noaa-isd') 

# save as a dataset 
files_dset = Dataset.Files.from_files('/tmp/noaa-isd').register(ws, 'my-delta-table') 

# make a sql-readable table
spark.sql("CREATE TABLE noaaisd USING DELTA LOCATION '/tmp/noaa-isd'")

## Using SQL Query Syntax
Data scientists will be delighted by support for SQL on Spark with Azure ML! 

Note this requires creating a "local" table - delta tables can be used. 

In [ ]:
%%synapse sql

# use SQL syntax to query/visulize data
SELECT * FROM noaaisd

## Model training

For classical learning, Apache Spark or Dask on Spark are decent options for scalability on the same cluster. 

For specialized ML hardware or orchestration in ML pipelines, Azure ML Compute is recommended.

Simply pass in the prepared Datasets to your training script like normal.

In [ ]:
from azureml.core import Experiment

script_folder = './pytorch-dnn'
os.makedirs(script_folder, exist_ok=True)

exp = Experiment(workspace=ws, name='keras-mnist')

script_params = {
    '--epochs': 1000,
    '--final_layer': 'sigmoid',
    '--X_dataset': ws.datasets['noaa-isd-X'],
    '--Y_dataset': ws.datasets['noaa-isd-Y']
}

est = Estimator(source_directory = '.', 
                script_params    = script_params, 
                compute_target   = ws.compute_targets['gpu-cluster'],
                entry_script     = 'keras_train.py', 
                use_docker       = False
                )

run = exp.submit(est)
RunDetails(run).show()

## Stop Session
If you're too eager to wait for the `idle-timeout` to do its job, you can manually cancel the session. 

In [ ]:
%synapse stop